## RTC+B adder model dataset

Targets: all 6 post-RTC+B adders (`RTRDPA`, `RTRDPRU`, `RTRDPRD`, `RTRDPRRS`, `RTRDPECRS`, `RTRDPNS`).
Predictors: the non-price predictors of `01.1_model_data_v2` (outage-aware `spare_capacity`).
No `price_wo_adders` / power price in this dataset.

Scope: 2025-12-05 HE1 (RTC+B go-live) onward. The window ends at the earliest input end date,
since rows missing any predictor are dropped.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# update these paths as each input is extended past its current end date
ADDER_PATH = '../../01_data/3_analysis/price_adders_analysis/post_202512/post_202512_price_adders_hourly_20251205_20261024.csv'

GEN_PATH = '../../01_data/2_cleaned/generation/hourly_solar_wind_generation_2020_2025.parquet'
LOAD_PATH = '../../01_data/2_cleaned/load/total_load_20201231_20261024.csv'
CAPACITY_PATH = '../../01_data/2_cleaned/generation/ERCOT nonRE capacity 2020-2025.csv'
NG_PATH = '../../01_data/3_analysis/ng_price/ng_prices_monthly.csv'
TEMP_PATHS = [
    '../../01_data/2_cleaned/weather/Texas_hourly_temp_2021_2025.csv',
    '../../01_data/2_cleaned/weather/Texas_hourly_temp_20260101_20261021.csv',
]
PEAK_HR_PATH = '../../01_data/2_cleaned/peak hours/peak_hour_2021_2025.csv'
OUTAGE_PATH = '../../01_data/2_cleaned/outage/archive_1h_ahead_outage_20210101_2026_1028.parquet'

RTCB_START = pd.Timestamp('2025-12-05 01:00')
ADDER_COLS = ['RTRDPA', 'RTRDPRU', 'RTRDPRD', 'RTRDPRRS', 'RTRDPECRS', 'RTRDPNS']

OUT_DIR = Path('../../01_data/4_model')

In [2]:
df_adder = pd.read_csv(ADDER_PATH, parse_dates=['date']).rename(columns={'date': 'datetime'})
df_adder = df_adder[df_adder['datetime'] >= RTCB_START]

df_gen = pd.read_parquet(GEN_PATH)

df_load = pd.read_csv(LOAD_PATH, parse_dates=['datetime'])
# perc_load: share of the calendar year's max load, computed on the full load series
# (not just the RTC+B window); 2026 is a partial year, so its max is year-to-date
df_load['perc_load'] = df_load['total_load'] / df_load.groupby(df_load['datetime'].dt.year)['total_load'].transform('max')

df_capacity = pd.read_csv(CAPACITY_PATH)
df_total_cap = df_capacity.filter(['year', 'total_mw'])
df_total_cap = df_total_cap.rename(columns={'total_mw': 'non_re_capacity'})
df_total_cap['non_re_capacity'] = df_total_cap['non_re_capacity'].astype(int)

df_ng = pd.read_csv(NG_PATH)
df_ng['date'] = pd.to_datetime(df_ng['date'])

df_temp = pd.concat([pd.read_csv(p, parse_dates=['datetime']) for p in TEMP_PATHS], ignore_index=True)

df_peak = pd.read_csv(PEAK_HR_PATH, parse_dates=['datetime'])

df_outage = pd.read_parquet(OUTAGE_PATH)
df_outage['datetime'] = pd.to_datetime(df_outage['datetime'])

### Input coverage
Each input's last timestamp; the earliest one bounds the dataset.

In [3]:
coverage = pd.DataFrame([
    ('adders', df_adder['datetime'].max()),
    ('renewable_gen', df_gen['datetime'].max()),
    ('total_load', df_load['datetime'].max()),
    ('non_re_capacity', pd.Timestamp(f"{df_total_cap['year'].max()}-12-31 23:00")),
    ('ng_price_monthly', df_ng['date'].max() + pd.offsets.MonthEnd(0)),
    ('tx_temp', df_temp['datetime'].max()),
    ('peak_hour', df_peak['datetime'].max()),
    ('outage', df_outage['datetime'].max()),
], columns=['input', 'last_timestamp']).sort_values('last_timestamp')
coverage

,input,last_timestamp
3,non_re_capacity,2025-12-31 23:00:00
6,peak_hour,2025-12-31 23:00:00
1,renewable_gen,2026-01-01 00:00:00
4,ng_price_monthly,2026-02-28 00:00:00
5,tx_temp,2026-10-22 00:00:00
0,adders,2026-10-25 00:00:00
2,total_load,2026-10-25 00:00:00
7,outage,2026-10-28 12:00:00


### Merge
Adders are the base table; predictors are left-joined so missing coverage shows up as NULLs
before the drop. Predictor definitions match `01.1_model_data_v2`.

In [4]:
import duckdb

con = duckdb.connect()
df_merged = con.execute("""
   WITH Merged AS (
      SELECT
         a.*
         ,MONTH(a.datetime) AS month
         ,renewable_gen
         ,total_load
         ,perc_load
         ,non_re_capacity
         ,henry_hub AS ng_price_monthly
         ,tx_temp
         ,peak_hour
         ,totalResourceMW AS outage

      FROM df_adder a
         LEFT JOIN df_gen g ON g.datetime = a.datetime
         LEFT JOIN df_load l ON l.datetime = a.datetime
         LEFT JOIN df_total_cap c ON YEAR(a.datetime) = c.year
         LEFT JOIN df_ng n ON YEAR(n.date) = YEAR(a.datetime)
                           AND MONTH(n.date) = MONTH(a.datetime)
         LEFT JOIN df_temp t ON t.datetime = a.datetime
         LEFT JOIN df_peak p ON p.datetime = a.datetime
         LEFT JOIN df_outage o ON o.datetime = a.datetime
      )

   SELECT
         datetime
         ,YEAR(datetime) AS year
         ,HOUR(datetime) AS time_of_day
         ,peak_hour
         ,CASE WHEN month IN (12, 1, 2)  THEN 'winter'
               WHEN month IN (3, 4, 5)   THEN 'spring'
               WHEN month IN (6, 7, 8)   THEN 'summer'
               WHEN month IN (9, 10, 11) THEN 'fall'
          END AS season
         ,total_load
         ,perc_load
         ,(total_load - renewable_gen)/(non_re_capacity - outage) AS spare_capacity
         ,ng_price_monthly
         ,ABS(tx_temp - 65) AS degree_days
         ,(CASE WHEN (tx_temp - 65) > 0 THEN TRUE
               WHEN (tx_temp - 65) < 0 THEN FALSE
               ELSE NULL END)
               AS heating_dd
         ,RTRDPA, RTRDPRU, RTRDPRD, RTRDPRRS, RTRDPECRS, RTRDPNS
   FROM Merged
   ORDER BY datetime;
""").fetchdf()

n_before = len(df_merged)
print('missing per column before drop:')
print(df_merged.isna().sum()[lambda s: s > 0])

df_model = df_merged.dropna().reset_index(drop=True)
print(f'\nrows: {n_before} -> {len(df_model)} after dropna')
print(f"window: {df_model['datetime'].min()} -> {df_model['datetime'].max()}")

missing per column before drop:
peak_hour           7128
spare_capacity      7128
ng_price_monthly    5712
degree_days           73
heating_dd            73
dtype: int64

rows: 7775 -> 647 after dropna
window: 2025-12-05 01:00:00 -> 2025-12-31 23:00:00


### Data inspection and export

In [5]:
summary = pd.DataFrame({
    'col_name': df_model.columns,
    'dtype': df_model.dtypes.values,
    'n_missing': df_model.isna().sum().values,
    'n_unique': df_model.nunique().values,
    'sample_values': [list(df_model[col].unique()[0:3]) for col in df_model.columns],
})
summary

,col_name,dtype,n_missing,n_unique,sample_values
0,datetime,datetime64[us],0,647,"[2025-12-05 01:00:00, 2025-12-05 02:00:00, 202..."
1,year,int64,0,1,[2025]
2,time_of_day,int64,0,24,"[1, 2, 3]"
3,peak_hour,boolean,0,2,"[False, True]"
4,season,str,0,1,[winter]
5,total_load,float64,0,646,"[50627.67, 49793.24, 49457.86]"
6,perc_load,float64,0,646,"[0.6035870673581949, 0.5936389272084368, 0.589..."
7,spare_capacity,float64,0,647,"[0.7914347082377637, 0.7751202598489528, 0.774..."
8,ng_price_monthly,float64,0,1,[4.263809523809524]
9,degree_days,float64,0,647,"[24.31789, 24.523113000000002, 25.1233]"


In [6]:
df_model[ADDER_COLS].describe()

,RTRDPA,RTRDPRU,RTRDPRD,RTRDPRRS,RTRDPECRS,RTRDPNS
count,647.000000,647.000000,647.000000,647.000000,647.000000,647.000000
mean,0.057067,0.343338,0.544227,0.245089,0.355835,1.254525
std,0.722919,1.333223,0.941889,1.051627,1.314692,3.019747
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,0.000000,0.000000,0.110000,0.013750,0.012500,0.150000
50%,0.000000,0.017500,0.300000,0.032500,0.052500,0.255000
75%,0.000000,0.257500,0.598750,0.131250,0.232500,1.035000
max,17.142500,26.030000,10.605000,21.025000,25.325000,37.340000


In [7]:
end_tag = df_model['datetime'].max().strftime('%Y%m%d')
out_path = OUT_DIR / f'rtcb_adder_model_dataset_20251205_{end_tag}.csv'
df_model.to_csv(out_path, index=False)
print('wrote', out_path, df_model.shape)

wrote ../../01_data/4_model/rtcb_adder_model_dataset_20251205_20251231.csv (647, 17)
